# Day 1: Encoding as Interpretation
**Date:** Sunday 21 June 2026  
**Corpus:** Beregovski *Jewish Instrumental Folk Music* — Malin & Shanahan (2025), *MTO* 31(3)


```{admonition} Conceptual check — before you code
:class: tip

Answer the self-assessment questions for Day 1 before running the cells below.
Questions open in a new tab — come back here when you're done.

**[→ Open Day 1 Quiz](../quizpages/day1_quiz.md)**
```


## The central argument

Encoding a score is not transcription — it is interpretation.
Every representational choice is a theory about what matters about music.
DARMS, MUSTRAN, and Humdrum encode three different philosophies;
the Beregovski kern corpus encodes a fourth set of decisions made by Malin and Shanahan.

Before we write any code, let's test your prior knowledge and assumptions.


---
## Part 1: Reading a Kern File

The encoding is the first analytical act. Read one file as plain text
before extracting anything from it.


In [ ]:
import requests, zipfile
from pathlib import Path
from collections import Counter
from itertools import islice

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from music21 import converter, note, interval

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.figsize'] = (10, 4)
print('Imports OK.')

In [ ]:
CORPUS_DIR = Path('beregovski_corpus')
KERN_DIR = CORPUS_DIR / 'kern'

if KERN_DIR.exists() and len(list(KERN_DIR.glob('*.krn'))) > 0:
    print(f'Corpus ready: {len(list(KERN_DIR.glob("*.krn")))} files.')
else:
    print('Downloading corpus from GitHub...')
    CORPUS_DIR.mkdir(exist_ok=True)
    r = requests.get('https://github.com/shanahdt/mode_in_klezmer/archive/refs/heads/main.zip')
    zp = CORPUS_DIR / 'repo.zip'
    zp.write_bytes(r.content)
    import shutil
    with zipfile.ZipFile(zp) as z: z.extractall(CORPUS_DIR)
    src = list(CORPUS_DIR.glob('mode_in_klezmer-*/kern'))
    if src:
        if KERN_DIR.exists(): shutil.rmtree(KERN_DIR)
        shutil.copytree(src[0], KERN_DIR)
        zp.unlink()
    print(f'Done. {len(list(KERN_DIR.glob("*.krn")))} kern files ready.')

In [ ]:
KERN_DIR = Path('beregovski_corpus/kern')
kern_files = sorted(KERN_DIR.glob('*.krn'))
print(f'File: {kern_files[0].name}')
print('=' * 50)
print(kern_files[0].read_text())

### Quick kern reference

| Symbol | Meaning |
|--------|---------|
| `!!!` | Global metadata record |
| `**kern` | Spine type |
| `*M6/8` | Meter |
| `*G:` | Key of G |
| `4g` | Quarter note G4 |
| `gg` | G5 (repeated letter = octave up) |
| `=1` | Barline |
| `*-` | End of spine |

```{note}
All tunes are notated in G regardless of original performance pitch.
This means pitch class data and scale-degree data are equivalent throughout the course.
```


---
## Part 2: Loading the Full Corpus

The `(df, streams)` pair is our core data structure throughout all 8 sessions.


In [ ]:
def load_corpus(kern_dir=KERN_DIR, verbose=True):
    pc2d = {7:1,9:2,11:3,0:4,2:5,4:6,6:7,8:2,10:3,1:4,3:5,5:6}
    records, sdict = {}, {}
    files = sorted(Path(kern_dir).glob('*.krn'))
    for i, f in enumerate(files):
        if verbose and i % 50 == 0: print(f'  {i+1}/{len(files)}...')
        try:
            s = converter.parse(str(f))
            ns = [n for n in s.flat.notes if isinstance(n, note.Note)]
            pcs = [n.pitch.pitchClass for n in ns]
            records[f.stem] = {
                'tune_id': f.stem, 'n_notes': len(ns),
                'pitches': [n.nameWithOctave for n in ns],
                'pitch_classes': pcs,
                'scale_degrees': [pc2d.get(p, 0) for p in pcs],
                'intervals': [interval.Interval(ns[j], ns[j+1]).semitones
                               for j in range(len(ns)-1)]
            }
            sdict[f.stem] = s
        except: pass
    if verbose: print(f'Loaded {len(records)} tunes.')
    return pd.DataFrame(records.values()), sdict

def get_ngrams(seq, n):
    return list(zip(*[islice(seq, i, None) for i in range(n)]))

print('Helper functions defined.')

In [ ]:
print('Loading corpus...')
df, streams = load_corpus()
try:
    meta = pd.read_csv('https://raw.githubusercontent.com/shanahdt/mode_in_klezmer/main/metadata.csv')
    df = df.merge(meta, on='tune_id', how='left')
    print(f'Metadata joined. {len(df)} tunes, columns: {df.columns.tolist()}')
except Exception as e:
    print(f'Metadata not loaded ({e}). Proceeding without mode labels.')

In [ ]:
# Basic corpus statistics
print(f'Tunes: {len(df)}')
print(f'Total notes: {df["n_notes"].sum():,}')
print(f'Mean notes/tune: {df["n_notes"].mean():.1f}')
df[['tune_id','n_notes']].head(10)

In [ ]:
# Pitch class profile — full corpus
all_pcs = [pc for pcs in df['pitch_classes'] for pc in pcs]
pc_counts = Counter(all_pcs)
total = sum(pc_counts.values())
pc_profile = [pc_counts.get(i,0)/total for i in range(12)]
pc_names = ['C','C#','D','Eb','E','F','F#','G','Ab','A','Bb','B']

fig, ax = plt.subplots()
ax.bar(pc_names, pc_profile, color='steelblue', edgecolor='white')
ax.set_xlabel('Pitch class')
ax.set_ylabel('Proportion')
ax.set_title('Pitch class profile — full corpus')
plt.tight_layout(); plt.show()
print('Which pitch class is most common, and why?')

---
## Day 1 Exercise: Encoding Audit

```{admonition} Exercise
Pick one tune. Read its kern file as plain text.
Identify **three encoding decisions** the kern file makes that a performer would not need to make.
For each: (1) what musical information is *fixed*, and (2) what is *left out*?
```


In [ ]:
# Choose a tune ID from the list below
print('Available tune IDs:')
print(df['tune_id'].tolist()[:20], '...')


In [ ]:
MY_TUNE = df['tune_id'].iloc[0]  # <-- change this
kern_file = KERN_DIR / f'{MY_TUNE}.krn'
print(f'Raw kern: {MY_TUNE}')
print('=' * 50)
print(kern_file.read_text())

### Your encoding audit

**Tune chosen:** *(fill in)*

**Decision 1:** What it fixes: / What it leaves out:

**Decision 2:** What it fixes: / What it leaves out:

**Decision 3:** What it fixes: / What it leaves out:


---
## Project Log — Entry 1

> *My corpus is [X — a subset: mode / genre / region / instrument].*  
> *My research question is [Y].*  
> *I expect [Z] because [musical or theoretical reasoning].*  
> *One encoding decision I would have made differently is [X] because [reason].*

*(Write here — 100–150 words)*


---
## Preview: Day 2 (Monday 22 Jun)

**Optional preparation:** Visit https://shanahdt.github.io/mode_in_klezmer/ and listen to
one freygish tune and one minor tune. Before looking at any data, write down what you
hear as the difference between them.
